# 08_screen_npass — 천연물(NPASS) 스크리닝

**한 줄 요약:** 07에서 저장한 모델로 NPASS 천연물 **9만여 개**의 활성 확률을 예측하고, 알려진 active와의 **유사도**도 함께 계산해 상위 후보를 뽑는다.
**용어:** active_prob=활성 확률 / max_sim_known=알려진 active와 최대 유사도(신규성 판단) / is_known=이미 아는 물질인지.
**큰 흐름:** ① 모델 불러오기 → ② 지문 함수 → ③ 알려진 active 로드 → ④ NPASS 지문·유사도 → ⑤ 배치 예측 → ⑥ 상위 저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + 저장된 모델 불러오기
라이브러리를 가져오고, 07에서 파일로 저장한 학습 모델(.pkl)을 다시 읽어온다.

In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

MODEL = "data/HSD17B13_screen_model.pkl"
NPASS = "data/npass_structures.tsv"
TRAIN = "data/HSD17B13_train_with_decoys.xlsx"
OUT_XLSX = "data/HSD17B13_npass_hits.xlsx"
OUT_CSV = "data/HSD17B13_npass_ranked_full.csv"
TOP_N = 300

with open(MODEL, "rb") as f:
    bundle = pickle.load(f)
model, fp_name, nbits = bundle["model"], bundle["fp_name"], bundle["nbits"]
print(f"모델: fingerprint={fp_name}, {nbits}bit "
      f"(학습 active {bundle['n_active']} / inactive {bundle['n_inactive']})")

🔎 **코드 뜯어보기 (셀 1)**
- `import pickle` : 파이썬 객체(모델)를 **파일로 저장/복원**하는 도구.
- `with open(MODEL, "rb") as f: bundle = pickle.load(f)` : 모델 파일을 **읽기+바이너리(rb)** 로 열어 복원. `with`=끝나면 자동으로 닫음.
- `model, fp_name, nbits = bundle["model"], bundle["fp_name"], bundle["nbits"]` : 딕셔너리에서 값 3개를 **한 번에** 각 변수로 꺼내기(언패킹).

### 셀 2 — 지문(fingerprint) 함수 준비
학습 때와 **똑같은** 방식으로 분자를 지문으로 바꾸는 함수를 만든다(안 그러면 예측이 어긋남).

In [ ]:
gens = {
    "ECFP4": rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=nbits),
    "RDKit": rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=nbits),
    "AtomPair": rdFingerprintGenerator.GetAtomPairGenerator(fpSize=nbits),
}
sim_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def maccs_np(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def featurize(mol):
    if fp_name == "MACCS":
        return maccs_np(mol)
    return gens[fp_name].GetFingerprintAsNumPy(mol)

🔎 **코드 뜯어보기 (셀 2)** *(생성기·maccs_np는 04에서 설명)*
- `def featurize(mol): if fp_name == "MACCS": return maccs_np(mol); return gens[fp_name].GetFingerprintAsNumPy(mol)` : 모델이 쓴 지문 종류(fp_name)에 맞춰 변환. `gens[fp_name]`=딕셔너리에서 해당 생성기 꺼내기.

### 셀 3 — 알려진 active 로드 (유사도 비교용)
학습에 쓴 실측 active들의 지문을 모아, 나중에 후보와의 유사도를 잰다.

In [ ]:
tr = pd.read_excel(TRAIN)
act_smis = tr[(tr.label == 1) & (tr.source == "real")]["canonical_smiles"].tolist()
act_fps, act_keep = [], []
for s in act_smis:
    m = Chem.MolFromSmiles(str(s))
    if m:
        act_fps.append(sim_gen.GetFingerprint(m))
        act_keep.append(s)
known_canon = set(act_keep)
print(f"알려진 active {len(act_fps)}개 로드(유사도 비교용)")

🔎 **코드 뜯어보기 (셀 3)**
- `tr[(tr.label == 1) & (tr.source == "real")]` : 라벨 1(active)이고 출처가 실측인 행만 고르기(**&**=그리고).
- `["canonical_smiles"].tolist()` : 그 열을 파이썬 **리스트**로. `set(act_keep)`=중복 없는 집합(빠른 포함 검사용).

### 셀 4 — NPASS 읽고 각 분자 지문·유사도 계산
천연물 9만개를 하나씩 지문으로 바꾸고, 알려진 active와 가장 닮은 정도를 계산한다.

In [ ]:
npass = pd.read_csv(NPASS, sep="\t", usecols=["np_id", "SMILES"])
print(f"NPASS {len(npass)}개 로드 → featurize 시작")

ids, canons, mats, sims, nn_smi, is_known = [], [], [], [], [], []
t0 = time.time()
for i, (npid, smi) in enumerate(zip(npass["np_id"], npass["SMILES"])):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    canon = Chem.MolToSmiles(mol)
    mats.append(featurize(mol))
    q = sim_gen.GetFingerprint(mol)
    s = DataStructs.BulkTanimotoSimilarity(q, act_fps)
    j = int(np.argmax(s))
    ids.append(npid)
    canons.append(canon)
    sims.append(float(s[j]))
    nn_smi.append(act_keep[j])
    is_known.append(canon in known_canon)
    if (i + 1) % 10000 == 0:
        print(f"  {i+1}/{len(npass)} 처리 ({time.time()-t0:.0f}s)")

X = np.vstack(mats).astype(np.float32)
print(f"featurize 완료: {X.shape[0]}개 ({time.time()-t0:.0f}s) → 배치 예측")

🔎 **코드 뜯어보기 (셀 4)**
- `pd.read_csv(NPASS, sep="\t", usecols=["np_id","SMILES"])` : 탭 구분 파일에서 필요한 두 열만 읽기.
- `for i, (npid, smi) in enumerate(zip(...))` : 번호와 (id, SMILES)를 함께 반복.
- `Chem.MolToSmiles(mol)` : 표준 SMILES. `DataStructs.BulkTanimotoSimilarity(q, act_fps)` : 한 지문과 active 전체의 유사도 목록.
- `np.argmax(s)` : 목록에서 **가장 큰 값의 위치(번호)**. `canon in known_canon` : 이미 아는 물질인지 True/False.

### 셀 5 — 한 번에 예측(배치)
모아둔 지문 전체를 **한 번에** 모델에 넣어 활성 확률을 구하고 결과 표로 정리한다.

In [ ]:
prob = model.predict_proba(X)[:, 1]

res = pd.DataFrame({
    "np_id": ids,
    "canonical_smiles": canons,
    "active_prob": prob,
    "max_sim_known": sims,
    "nearest_active": nn_smi,
    "is_known": is_known,
}).sort_values("active_prob", ascending=False).reset_index(drop=True)

res.to_csv(OUT_CSV, index=False)

🔎 **코드 뜯어보기 (셀 5)**
- `np.vstack(mats).astype(np.float32)` : 지문들을 표로 쌓고 실수형으로. **배치 예측**을 위해 전체를 한 행렬로.
- `model.predict_proba(X)[:, 1]` : 전 분자의 예측 확률을 **한 번에** 계산, `[:,1]`=active일 확률.
- `pd.DataFrame({...}).sort_values("active_prob", ascending=False)` : 결과 표를 확률 높은 순으로 정렬.

### 셀 6 — 신규 상위 후보 저장 + 요약
이미 알려진 물질을 빼고 상위 후보를 엑셀로 저장하고 요약을 출력한다.

In [ ]:
novel = res[~res["is_known"]].head(TOP_N)
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    novel.to_excel(w, sheet_name=f"top{TOP_N}_novel", index=False)
    res.head(50).to_excel(w, sheet_name="top50_all", index=False)

print(f"\n전체 순위: {OUT_CSV}")
print(f"상위 후보(신규): {OUT_XLSX}")
print(f"\nactive_prob >= 0.9: {(res.active_prob>=0.9).sum()}개 "
      f"(그중 신규 {((res.active_prob>=0.9)&(~res.is_known)).sum()}개)")
print(f"NPASS에서 발견된 '이미 알려진 active': {res.is_known.sum()}개")
print("\n=== 신규 후보 상위 15 ===")
print(novel.head(15)[["np_id", "active_prob", "max_sim_known"]].to_string(index=False))

🔎 **코드 뜯어보기 (셀 6)**
- `res[~res["is_known"]]` : `~`=부정 → 이미 아는 게 **아닌**(신규) 것만. `.head(TOP_N)`=상위 N개.
- `with pd.ExcelWriter(...) as w:` 안에서 `to_excel(w, sheet_name=...)` 두 번 → **시트 2개**로 저장.
- `(res.active_prob>=0.9).sum()` : 확률 0.9 이상 개수. `f"...{...}..."`=f-문자열.